# Transformers: replicando *Attention Is All You Need*

En la clase 08 el decoder recibía del encoder un único estado $(h, c)$ de tamaño fijo, y en las oraciones largas la traducción se degradaba. El mecanismo de **atención** resuelve ese cuello de botella: en cada paso, el decoder consulta todos los estados del encoder y decide cuánto peso darle a cada uno. El Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) va un paso más allá: elimina la recurrencia y arma el encoder y el decoder solo con atención y capas densas. Así procesa todas las posiciones de una oración en paralelo.

En esta clase replicamos el paper sobre el mismo problema de la clase 08, la traducción de inglés a español. Cada sección cita el fragmento del paper que describe un componente y a continuación lo implementa.

<div align="center">
    <img src="https://d1.awsstatic.com/GENAI-1.151ded5440b4c997bac0642ec669a00acff2cca1.png" width="400px">
</div>

## Introducción

### Objetivos

1. **Implementar desde cero los componentes del Transformer** a partir del paper: positional encoding, multi-head attention, máscaras, encoder y decoder.
2. **Entender cómo se entrena en paralelo** un modelo que genera de forma autorregresiva: el desplazamiento del target y la máscara *look-ahead*.
3. **Entrenar un traductor inglés-español** con el optimizador y la regularización del paper.
4. **Evaluar las traducciones con BLEU**, compararlas con las del seq2seq de la clase 08 y visualizar la atención.

### Contenido

1. Carga del dataset, división, vocabularios y `DataLoader` (como en la clase 08).
2. Positional encoding.
3. Scaled dot-product attention y multi-head attention.
4. Máscaras de padding y *look-ahead*.
5. Feed-forward, encoder y decoder.
6. Entrenamiento con el *learning rate schedule* del paper.
7. Generación de traducciones, atención cruzada y evaluación en test con BLEU.

### Sobre el conjunto de datos

Usamos los mismos pares inglés-español de [Tatoeba](https://tatoeba.org/es/downloads) que en la clase 08 (licencia CC BY 2.0 FR). Esta vez nos quedamos con las oraciones de hasta 15 palabras en los dos idiomas, casi el doble que en la clase 08.

### Referencias

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762), Vaswani et al. (2017).
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/), Jay Alammar: encoder y decoder, como en esta clase.
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) y [LLM Visualization](https://bbycroft.net/llm): visualizaciones interactivas de modelos *decoder-only* (tipo GPT).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence  # rellena secuencias de distinto largo hasta el largo de la más larga

from torchinfo import summary  # resumen de capas, formas y parámetros

import copy  # copiar los pesos de la mejor época
import math
import random
import re  # expresiones regulares para la limpieza de texto
import time
from collections import Counter  # contar la frecuencia de cada palabra (vocabulario)

import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu  # métrica BLEU para traducción automática

from utils import plot_training

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
random.seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

## Carga de datos y preprocesamiento

La carga, la limpieza, la división y los vocabularios son los de la clase 08; solo cambia el largo máximo de las oraciones.

In [ ]:
def clean_text(text):
    # Convertimos a minúsculas
    text = text.lower()

    # Insertamos espacios alrededor de los símbolos de puntuación que queremos conservar
    text = re.sub(r"([¿?¡!])", r" \1 ", text)

    # Eliminamos todo lo que no sea letras, números o los símbolos que queremos conservar
    # (el apóstrofo también se elimina: "i'm" queda como "i m")
    text = re.sub(r"[^a-zA-Z0-9áéíóúüñ¿?¡!]+", " ", text)

    # Eliminamos los espacios extra
    text = re.sub(r"\s+", " ", text).strip()

    return text


print(clean_text("I'm sure you're right!"))
print(clean_text("¿Cómo estás?"))

In [ ]:
DATA_PATH = "data/English-Spanish.tsv"
MAX_SENTENCE_LENGTH = 15  # máximo de palabras por oración, en los dos idiomas


def load_data(source_file, max_words):
    src_texts, trg_texts = [], []
    with open(source_file, "r", encoding="utf-8") as f:
        for line in f:
            elements = line.split("\t")
            src = clean_text(elements[1])  # inglés
            trg = clean_text(elements[3])  # español

            # Nos quedamos con los pares en los que las dos oraciones tienen hasta max_words palabras
            if len(src.split()) <= max_words and len(trg.split()) <= max_words:
                src_texts.append(src)
                trg_texts.append(trg)
    return src_texts, trg_texts


src_texts, trg_texts = load_data(DATA_PATH, MAX_SENTENCE_LENGTH)
print(f"Cantidad de pares: {len(src_texts)}")

In [ ]:
for idx in random.sample(range(len(src_texts)), 8):
    print(f"Inglés:  {src_texts[idx]}")
    print(f"Español: {trg_texts[idx]}\n")

### División en entrenamiento, validación y test

Como en la clase 08, separamos 80% para entrenamiento, 10% para validación y 10% para test **antes** de construir los vocabularios. El test se usa una sola vez, al final.

<small>Nota: como en la clase 08, muchas oraciones en inglés tienen varias traducciones y una parte de las oraciones de validación y test también aparece en entrenamiento, con otra traducción. La evaluación es algo optimista.</small>

In [ ]:
pairs = list(zip(src_texts, trg_texts))
train_pairs, val_pairs, test_pairs = random_split(
    pairs, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(SEED)
)
print(f"Entrenamiento: {len(train_pairs)} | Validación: {len(val_pairs)} | Test: {len(test_pairs)}")

## Vocabularios

Un vocabulario por idioma, solo con el conjunto de entrenamiento, con los tokens especiales `<PAD>`, `<SOS>`, `<EOS>` y `<UNK>` de la clase 08.

In [ ]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
FREQ_THRESHOLD = 3  # frecuencia mínima para incluir una palabra en el vocabulario


class Vocab:
    def __init__(self):
        # mapea palabras a índices
        self.word2index = {}
        # mapea índices a palabras
        self.index2word = {}
        # autonumeración de índices
        self.index = 0

        # Tokens especiales
        self.add_special_tokens()

    def add_special_tokens(self):
        self.add_word(PAD_TOKEN)
        self.add_word(SOS_TOKEN)
        self.add_word(EOS_TOKEN)
        self.add_word(UNK_TOKEN)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.index
            self.index2word[self.index] = word
            self.index += 1

    def build_vocab(self, sentences, min_freq=1):
        word_counter = Counter()
        for sentence in sentences:
            for word in sentence.split():
                word_counter[word] += 1

        # Filtrar palabras que no alcanzan la frecuencia mínima
        words = [word for word, count in word_counter.items() if count >= min_freq]

        # Agregar palabras filtradas al vocabulario
        for word in words:
            self.add_word(word)

    def __len__(self):
        return len(self.word2index)

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.index2word.get(key, UNK_TOKEN)
        if isinstance(key, str):
            return self.word2index.get(key, self.word2index[UNK_TOKEN])


# Construimos los vocabularios solo con el conjunto de entrenamiento
SRC_VOCAB = Vocab()
TRG_VOCAB = Vocab()

SRC_VOCAB.build_vocab([src for src, _ in train_pairs], min_freq=FREQ_THRESHOLD)
TRG_VOCAB.build_vocab([trg for _, trg in train_pairs], min_freq=FREQ_THRESHOLD)

SRC_VOCAB_SIZE = len(SRC_VOCAB)
TRG_VOCAB_SIZE = len(TRG_VOCAB)

print(f"Tamaño del vocabulario en inglés: {SRC_VOCAB_SIZE}")
print(f"Tamaño del vocabulario en español: {TRG_VOCAB_SIZE}")

In [ ]:
# Función para codificar una frase (strings -> índices)
def encode_sentence(sentence, vocab):
    return [vocab[word] for word in sentence.split()]


print(encode_sentence("hello world", SRC_VOCAB))
print(encode_sentence("hola mundo extraterrestre", TRG_VOCAB))

In [ ]:
# Función para decodificar una secuencia de índices (índices -> strings)
def decode_sentence(indices, vocab):
    if isinstance(indices, torch.Tensor):  # en caso que nos pasen un tensor
        indices = indices.tolist()
    return " ".join(
        [
            vocab[idx]
            for idx in indices
            if idx != vocab[PAD_TOKEN]
            and idx != vocab[EOS_TOKEN]
            and idx != vocab[SOS_TOKEN]
        ]
    )


print(
    decode_sentence(
        [
            TRG_VOCAB[SOS_TOKEN],
            10,
            11,
            12,
            TRG_VOCAB[EOS_TOKEN],
            TRG_VOCAB[PAD_TOKEN],
            TRG_VOCAB[PAD_TOKEN],
        ],
        TRG_VOCAB,
    )
)

## Dataset y padding

El `Dataset` es el de la clase 08: al target le agregamos `<SOS>` al principio y `<EOS>` al final.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        super(TranslationDataset, self).__init__()
        self.pairs = pairs  # lista de pares (oración en inglés, oración en español)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        source_sentence, target_sentence = self.pairs[idx]
        encoded_source_sentence = encode_sentence(source_sentence, SRC_VOCAB)
        encoded_target_sentence = encode_sentence(target_sentence, TRG_VOCAB)

        # Añadimos tokens especiales <SOS> y <EOS>
        encoded_target_sentence = (
            [TRG_VOCAB[SOS_TOKEN]] + encoded_target_sentence + [TRG_VOCAB[EOS_TOKEN]]
        )

        x = torch.tensor(encoded_source_sentence, dtype=torch.long)
        y = torch.tensor(encoded_target_sentence, dtype=torch.long)

        return x, y


train_dataset = TranslationDataset(train_pairs)
val_dataset = TranslationDataset(val_pairs)
test_dataset = TranslationDataset(test_pairs)

In [ ]:
# Probamos algunos ejemplos de entrenamiento
for idx in range(3):
    x, y = train_dataset[idx]
    print(f"SRC:           {decode_sentence(x, SRC_VOCAB)}")
    print(f"SRC (índices): {x}")
    print(f"TRG:           {decode_sentence(y, TRG_VOCAB)}")
    print(f"TRG (índices): {y}\n")

En la clase 08 el source se rellenaba a la izquierda para que el último estado del encoder quedara después de palabras reales. El Transformer no tiene un último estado: cada posición tiene su propia salida, y una **máscara** le indica a la atención qué posiciones son `<PAD>` para que no las tenga en cuenta. Por eso rellenamos el source y el target a la derecha.

In [ ]:
def collate_fn(batch):
    # batch: lista de pares (source, target) devueltos por TranslationDataset
    sources, targets = zip(*batch)

    # Padding a la derecha en los dos: las máscaras se encargan de los <PAD>
    sources_padded = pad_sequence(
        sources, padding_value=SRC_VOCAB[PAD_TOKEN], batch_first=True, padding_side="right"
    )
    targets_padded = pad_sequence(
        targets, padding_value=TRG_VOCAB[PAD_TOKEN], batch_first=True, padding_side="right"
    )

    return sources_padded, targets_padded

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS
)

x, y = next(iter(train_loader))
print(f"Source batch shape: {x.shape}")  # (batch, largo de la oración más larga del batch)
print(f"Target batch shape: {y.shape}")  # hasta MAX_SENTENCE_LENGTH + 2, por <SOS> y <EOS>

## Modelo

> **Encoder**: The encoder is composed of a stack of $N = 6$ identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. [...]
>
> **Decoder**: The decoder is also composed of a stack of $N = 6$ identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. [...]

El modelo *base* del paper tiene 65 millones de parámetros y se entrenó 12 horas en 8 GPUs sobre 4.5 millones de pares de oraciones. Para entrenar en clase reducimos el tamaño:

| Hiperparámetro | Paper (base) | Clase |
|---|---|---|
| Capas del encoder y del decoder ($N$) | 6 | 3 |
| Dimensión del modelo ($d_{model}$) | 512 | 256 |
| Cabezas de atención ($h$) | 8 | 8 |
| Dimensión interna del feed-forward ($d_{ff}$) | 2048 | 1024 |
| Dropout ($P_{drop}$) | 0.1 | 0.1 |

In [ ]:
D_MODEL = 256  # dimensión de los embeddings y de todas las subcapas
NUM_LAYERS = 3  # capas del encoder y del decoder
HEADS = 8  # cabezas de atención (D_MODEL tiene que ser divisible por HEADS)
FORWARD_EXPANSION = 4  # d_ff = FORWARD_EXPANSION * D_MODEL
DROPOUT = 0.1
MAX_SENTENCE_LENGTH_MODEL = MAX_SENTENCE_LENGTH + 2  # tenemos en cuenta los tokens <SOS> y <EOS>

### Positional Encoding

<div style="text-align: center;">
    <img src="https://kazemnejad.com/img/transformer_architecture_positional_encoding/model_arc.jpg" width="700"/>
</div>

> Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the bottoms of the encoder and decoder stacks. The positional encodings have the same dimension $d_{model}$ as the embeddings, so that the two can be summed. [...] In this work, we use sine and cosine functions of different frequencies:
> $$
> \begin{aligned}
> \text{PE}_{(pos,2i)} &= \sin(pos / 10000^{2i/d_{\text{model}}}) \\
> \text{PE}_{(pos,2i+1)} &= \cos(pos / 10000^{2i/d_{\text{model}}})
> \end{aligned}
> $$
> where $pos$ is the position and $i$ is the dimension. That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$. We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

La atención no distingue el orden: si se permutan las palabras de una oración, cada palabra obtiene la misma salida, permutada. Por eso el orden se agrega a la entrada, sumando a cada embedding un vector que depende de su posición. Algunas propiedades de esta codificación:

- Cada posición tiene un vector distinto, y se calcula con una fórmula, sin parámetros entrenables.
- El par (seno, coseno) de una misma frecuencia funciona como las agujas de un reloj: pasar de la posición $pos$ a $pos + k$ es una rotación que depende solo de $k$, no de $pos$. Esa es la "función lineal" que menciona el paper ([explicación](https://blog.timodenk.com/linear-relationships-in-the-transformers-positional-encoding/)).
- La primera dimensión ($i = 0$) completa un ciclo cada $2\pi \approx 6.28$ posiciones; las últimas tienen ciclos de hasta $10000 \cdot 2\pi$ posiciones. Las dimensiones de frecuencia alta distinguen posiciones vecinas, y las de frecuencia baja, posiciones lejanas.
- Se puede calcular para cualquier posición, incluso más largas que las del entrenamiento. En la implementación precalculamos `max_len` posiciones.

**¿Por qué se suman y no se concatenan?** Concatenar aumentaría la dimensión de todas las capas. En la práctica, sumar funciona bien: el modelo aprende a usar subespacios distintos para el significado y para la posición ([Ke et al., 2020](https://arxiv.org/abs/2006.15595) analiza esta decisión).

Links útiles:
- [Transformer Architecture: The Positional Encoding](https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)
- [\[video\] ¿Por qué estas REDES NEURONALES son tan POTENTES?](https://www.youtube.com/watch?v=xi94v_jl26U)
- [\[video\] How do Transformer Models keep track of the order of words? Positional Encoding](https://www.youtube.com/watch?v=IHu3QehUmrQ)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super(PositionalEncoding, self).__init__()
        assert d_model % 2 == 0, "d_model debe ser par para usar sinusoides"

        # TODO:
        #   1. pe = torch.zeros((max_len, d_model))
        #   2. pos: posiciones 0..max_len-1 con forma [max_len, 1] (torch.arange + unsqueeze)
        #   3. two_i = torch.arange(0, d_model, 2) y div = 10_000 ** (two_i / d_model)
        #   4. columnas pares: pe[:, 0::2] = sin(pos / div); impares: pe[:, 1::2] = cos(pos / div)
        #   5. agregar la dimensión del batch: pe con forma [1, max_len, d_model]
        pe = None

        # 'pe' es un buffer: se guarda con el modelo y se mueve con .to(DEVICE), pero no se entrena
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        # pe: [1, max_len, d_model]
        # TODO: sumar a x las primeras seq_len posiciones de self.pe
        pass


summary(PositionalEncoding(D_MODEL, MAX_SENTENCE_LENGTH_MODEL))

No tiene parámetros entrenables. Con `d_model = 4` y 6 posiciones, el resultado tiene que ser:

```python
[[ 0.          1.          0.          1.        ]
 [ 0.84147096  0.54030234  0.00999983  0.99995   ]
 [ 0.9092974  -0.41614684  0.01999867  0.9998    ]
 [ 0.14112    -0.9899925   0.0299955   0.99955004]
 [-0.7568025  -0.6536436   0.03998933  0.9992001 ]
 [-0.9589243   0.2836622   0.04997917  0.99875027]]
```

Las dos primeras columnas ($i = 0$) cambian rápido de una posición a la siguiente; las dos últimas, lento.

In [ ]:
pos_encoding = PositionalEncoding(d_model=4, max_len=6)
print(pos_encoding.pe.squeeze(0).numpy())

### Multi-Head Attention

> We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension $d_k$, and values of dimension $d_v$. We compute the dot products of the query with all keys, divide each by $\sqrt{d_k}$, and apply a softmax function to obtain the weights on the values. [...] We compute the matrix of outputs as:
>
> $$
> \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
> $$
>
> [...] We suspect that for large values of $d_k$, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by $\frac{1}{\sqrt{d_k}}$.
>
> [...] Instead of performing a single attention function with $d_{\text{model}}$-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values $h$ times with different, learned linear projections to $d_k$, $d_k$ and $d_v$ dimensions, respectively. [...] These are concatenated and once again projected, resulting in the final values:
>
> $$
> \begin{aligned}
> \text{MultiHead}(Q, K, V) &= \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O \\
> \text{where head}_i &= \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
> \end{aligned}
> $$
>
> [...] we employ $h = 8$ parallel attention layers, or heads. For each of these we use $d_k = d_v = d_{model}/h = 64$.

La atención funciona como una búsqueda en un diccionario, pero "blanda". Cada posición arma una **query** (qué busca), y cada posición de la secuencia consultada ofrece una **key** (qué contiene) y un **value** (qué aporta). El producto escalar $q \cdot k$ mide cuánto coincide la query con cada key; el softmax convierte esos puntajes en pesos que suman 1, y la salida es el promedio de los values con esos pesos. En la clase 08 el decoder solo veía $(h, c)$; con atención, en cada paso combina todos los estados del encoder.

Las $h$ cabezas no agregan costo: en vez de una atención con vectores de $d_{model}$, el vector se corta en $h$ partes de $d_k = d_{model}/h$ y cada cabeza atiende por su cuenta, con sus propios pesos. En el código, las proyecciones $W^Q, W^K, W^V$ de todas las cabezas son una sola `nn.Linear(d_model, d_model)`, y el corte en cabezas es un `reshape`:

`[batch, len, d_model]` → `reshape` → `[batch, len, heads, d_k]` → `permute` → `[batch, heads, len, d_k]`

Los puntajes $QK^\top$ tienen forma `[batch, heads, queries_len, keys_len]`: una fila por query, una columna por key. Después de multiplicar por $V$ se hace el camino inverso para volver a `[batch, queries_len, d_model]`.

<div style="text-align: center;">
    <img src="https://yjucho1.github.io/assets/img/2018-10-13/transformer.png" width="900"/>
</div>

Links útiles:
- [\[AI by hand\] 11. Self Attention](https://aibyhand.substack.com/p/11-can-you-calculate-self-attention): calcular la atención a mano, paso a paso.
- [\[video\] ¿Qué es un TRANSFORMER?](https://www.youtube.com/watch?v=aL-EmKuB078)

> The Transformer uses multi-head attention in three different ways:
> - In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. [...]
> - The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. [...]
> - Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to $-\infty$) all values in the input of the softmax which correspond to illegal connections.

Por eso `MultiHeadAttention` recibe por separado `queries`, `keys` y `values`: en la self-attention los tres son el mismo tensor; en la atención encoder-decoder, las queries vienen del decoder y las keys y values, del encoder. Guardamos los pesos de atención en `self.attention_weights` para graficarlos al final de la clase.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# MultiHeadAttention                       --
# ├─Linear: 1-1                            65,792
# ├─Linear: 1-2                            65,792
# ├─Linear: 1-3                            65,792
# ├─Linear: 1-4                            65,792
# Total params: 263,168

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super(MultiHeadAttention, self).__init__()

        self.d_k = d_model // heads
        self.heads = heads
        self.d_model = d_model

        assert self.d_k * heads == d_model, "el d_model debe ser divisible por los heads"

        # TODO: cuatro capas nn.Linear(d_model, d_model): self.queries, self.keys, self.values
        # (W^Q, W^K, W^V de todas las cabezas juntas) y self.output (W^O)

        self.attention_weights = None  # pesos de la última llamada (para graficar)

    def forward(self, queries, keys, values, mask):
        # queries: [batch_size, queries_len, d_model]
        # keys:    [batch_size, keys_len, d_model]
        # values:  [batch_size, values_len, d_model]  (values_len = keys_len)
        # mask:    se puede expandir (broadcasting) a [batch_size, heads, queries_len, keys_len], o None
        batch_size = queries.size(0)
        queries_len, keys_len, values_len = queries.size(1), keys.size(1), values.size(1)

        # TODO:
        #   1. proyectar queries, keys y values con su capa lineal
        #   2. dividir en cabezas: reshape a [batch_size, len, heads, d_k] y
        #      permute(0, 2, 1, 3) a [batch_size, heads, len, d_k]
        #   3. scores = Q K^T / sqrt(d_k) con torch.matmul y keys.transpose(-2, -1)
        #      -> [batch_size, heads, queries_len, keys_len]
        #   4. si mask no es None: scores.masked_fill(mask == False, float("-inf"))
        #   5. attention = softmax de scores sobre la última dimensión (keys);
        #      guardar self.attention_weights = attention.detach()
        #   6. out = attention @ values -> [batch_size, heads, queries_len, d_k]
        #   7. unir las cabezas: permute(0, 2, 1, 3) y reshape a [batch_size, queries_len, d_model]
        #   8. devolver self.output(out)
        pass


summary(MultiHeadAttention(d_model=D_MODEL, heads=HEADS))

### Máscaras

La atención usa dos máscaras. En las dos, `True` indica una posición que se puede atender y `False`, una posición que se anula con $-\infty$ antes del softmax.

#### Máscara de padding del source

Cada token del source puede atender a todos los demás, salvo a los `<PAD>`. La máscara tiene forma `[batch, 1, 1, src_len]`: por broadcasting, la misma máscara vale para todas las cabezas y para todas las queries, y anula las columnas (keys) de padding.

In [ ]:
def create_src_mask(src, src_pad_idx):
    # src: [batch_size, src_len]
    src_mask = (src != src_pad_idx).unsqueeze(1).unsqueeze(2)
    # src_mask: [batch_size, 1, 1, src_len]
    return src_mask


src_example = torch.tensor([[5, 6, 7, 0, 0]])  # una oración de 3 tokens y 2 <PAD>
src_mask_example = create_src_mask(src_example, SRC_VOCAB[PAD_TOKEN])
print(src_mask_example)

scores_example = torch.rand((1, 1, 5, 5))  # puntajes de atención al azar: 5 queries x 5 keys
print(torch.softmax(scores_example.masked_fill(src_mask_example == False, float("-inf")), dim=-1))

Después del softmax, las columnas de `<PAD>` tienen peso 0 y cada fila sigue sumando 1.

#### Máscara *look-ahead* del target

> We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position $i$ can depend only on the known outputs at positions less than $i$.

En la self-attention del decoder, la posición $i$ no puede ver las posiciones siguientes, porque son justamente las palabras que tiene que predecir. La máscara es una matriz triangular inferior ([`torch.tril`](https://pytorch.org/docs/stable/generated/torch.tril.html)) combinada con la máscara de padding del target.

In [ ]:
def create_trg_mask(trg, trg_pad_idx):
    # trg: [batch_size, trg_len]
    trg_pad_mask = (trg != trg_pad_idx).unsqueeze(1).unsqueeze(2)
    # trg_pad_mask: [batch_size, 1, 1, trg_len]

    trg_len = trg.size(1)
    # device=trg.device: torch.full crea el tensor en la CPU si no se indica otro dispositivo
    trg_sub_mask = torch.tril(torch.full((trg_len, trg_len), True, device=trg.device))
    # trg_sub_mask: [trg_len, trg_len]

    trg_mask = trg_pad_mask & trg_sub_mask  # combinamos ambas máscaras (broadcasting)
    # trg_mask: [batch_size, 1, trg_len, trg_len]
    return trg_mask


trg_example = torch.tensor([[1, 8, 9, 2, 0]])  # <SOS> palabra palabra <EOS> <PAD>
print(create_trg_mask(trg_example, TRG_VOCAB[PAD_TOKEN]))

La fila $i$ indica qué posiciones ve la query $i$. La última fila corresponde a una query de `<PAD>`: su salida no importa, porque la pérdida ignora esas posiciones (`ignore_index`). Ninguna fila queda entera en `False`: si eso pasara, el softmax de una fila de $-\infty$ daría `NaN`.

### Feed-Forward

> In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully connected feed-forward network, which is applied to each position separately and identically. This consists of two linear transformations with a ReLU activation in between.
>
> $$
> \text{FFN}(x) = \max(0, xW_1 + b_1) W_2 + b_2
> $$
>
> [...] The dimensionality of input and output is $d_{\text{model}} = 512$, and the inner-layer has dimensionality $d_{ff} = 2048$.

La atención mezcla información entre posiciones; el feed-forward procesa cada posición por separado, con los mismos pesos. `nn.Linear` ya hace eso cuando recibe un tensor `[batch, len, d_model]`: se aplica sobre la última dimensión.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# FeedForward                              --
# ├─Sequential: 1-1                        --
# │    └─Linear: 2-1                       263,168
# │    └─ReLU: 2-2                         --
# │    └─Linear: 2-3                       262,400
# Total params: 525,568


class FeedForward(nn.Module):
    def __init__(self, d_model, expansion_factor):
        super(FeedForward, self).__init__()
        # TODO: Linear(d_model, expansion_factor * d_model) -> ReLU -> Linear(expansion_factor * d_model, d_model)
        pass

    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        pass


summary(FeedForward(d_model=D_MODEL, expansion_factor=FORWARD_EXPANSION))

### Encoder

> We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is $\text{LayerNorm}(x + \text{Sublayer}(x))$, where $\text{Sublayer}(x)$ is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension $d_{model} = 512$.
>
> [...] **Residual Dropout**: We apply dropout to the output of each sub-layer, before it is added to the sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of $P_{drop} = 0.1$.
>
> [...] In the embedding layers, we multiply those weights by $\sqrt{d_{model}}$.

Cada subcapa sigue el mismo patrón: `x = LayerNorm(x + Dropout(Subcapa(x)))`. Las conexiones residuales son las de la clase 05 y permiten apilar muchas capas. `nn.LayerNorm` normaliza cada vector de $d_{model}$ por separado (media 0 y varianza 1 sobre sus componentes), a diferencia de BatchNorm, que normaliza sobre el batch.

En este punto nos apartamos del paper: no multiplicamos el embedding por $\sqrt{d_{model}}$. Esa escala supone pesos inicializados con valores chicos, pero `nn.Embedding` los inicializa con una normal $N(0, 1)$, que ya tiene una escala comparable a la del positional encoding (valores entre -1 y 1). Multiplicarlos por $\sqrt{256} = 16$ haría que la posición casi no influyera; en nuestras pruebas, el BLEU en validación bajó de 45 a 37.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# EncoderLayer                             [256, 17, 256]            --
# ├─MultiHeadAttention: 1-1                [256, 17, 256]            --
# │    └─Linear: 2-1                       [256, 17, 256]            65,792
# │    └─Linear: 2-2                       [256, 17, 256]            65,792
# │    └─Linear: 2-3                       [256, 17, 256]            65,792
# │    └─Linear: 2-4                       [256, 17, 256]            65,792
# ├─Dropout: 1-2                           [256, 17, 256]            --
# ├─LayerNorm: 1-3                         [256, 17, 256]            512
# ├─FeedForward: 1-4                       [256, 17, 256]            --
# │    └─Sequential: 2-5                   [256, 17, 256]            --
# │    │    └─Linear: 3-1                  [256, 17, 1024]           263,168
# │    │    └─ReLU: 3-2                    [256, 17, 1024]           --
# │    │    └─Linear: 3-3                  [256, 17, 256]            262,400
# ├─Dropout: 1-5                           [256, 17, 256]            --
# ├─LayerNorm: 1-6                         [256, 17, 256]            512
# Total params: 789,760


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout, forward_expansion):
        super(EncoderLayer, self).__init__()
        # TODO: self.self_attention (MultiHeadAttention), self.norm1 y self.norm2 (nn.LayerNorm(d_model)),
        # self.feed_forward (FeedForward) y self.dropout (nn.Dropout)
        pass

    def forward(self, x, mask):
        # x: [batch_size, seq_len, d_model]
        # mask: [batch_size, 1, 1, seq_len]
        # TODO:
        #   1. self-attention: queries, keys y values son x (con la máscara)
        #   2. x = norm1(x + dropout(atención))
        #   3. x = norm2(x + dropout(feed_forward(x)))
        pass


rand_input = torch.randn((BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL, D_MODEL))  # batch de embeddings al azar
rand_mask = torch.ones((BATCH_SIZE, 1, 1, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)  # no enmascaramos nada

summary(
    EncoderLayer(d_model=D_MODEL, heads=HEADS, dropout=DROPOUT, forward_expansion=FORWARD_EXPANSION),
    input_data=(rand_input, rand_mask),
)

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# Encoder                                  [256, 17, 256]            --
# ├─Embedding: 1-1                         [256, 17, 256]            2,961,664
# ├─PositionalEncoding: 1-2                [256, 17, 256]            --
# ├─Dropout: 1-3                           [256, 17, 256]            --
# ├─ModuleList: 1-4                        --                        --
# │    └─EncoderLayer: 2-1                 [256, 17, 256]            789,760
# │    └─EncoderLayer: 2-2                 [256, 17, 256]            789,760
# │    └─EncoderLayer: 2-3                 [256, 17, 256]            789,760
# Total params: 5,330,944

class Encoder(nn.Module):
    def __init__(self, src_vocab_size, d_model, num_layers, heads, forward_expansion, dropout, max_length):
        super(Encoder, self).__init__()
        self.d_model = d_model
        # TODO: self.word_embedding (nn.Embedding con padding_idx=SRC_VOCAB[PAD_TOKEN]),
        # self.position_embedding (PositionalEncoding), self.dropout y
        # self.layers: nn.ModuleList con num_layers EncoderLayer
        pass

    def forward(self, x, mask):
        # x: [batch_size, seq_len] con índices enteros
        # TODO:
        #   1. embedding
        #   2. sumar el positional encoding y aplicar dropout
        #   3. pasar por cada capa de self.layers (con la máscara)
        pass


rand_input = torch.randint(0, SRC_VOCAB_SIZE, (BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL))  # batch de tokens al azar
rand_mask = torch.ones((BATCH_SIZE, 1, 1, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)

summary(
    Encoder(SRC_VOCAB_SIZE, D_MODEL, NUM_LAYERS, HEADS, FORWARD_EXPANSION, DROPOUT, MAX_SENTENCE_LENGTH_MODEL),
    input_data=(rand_input, rand_mask),
    depth=2,
)

### Decoder

> **Decoder**: The decoder is also composed of a stack of $N = 6$ identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization.

Cada capa del decoder tiene tres subcapas:

1. Self-attention con la máscara *look-ahead*: cada posición ve las palabras ya generadas.
2. Atención encoder-decoder (*cross-attention*): las queries vienen del decoder; las keys y los values, de la salida del encoder. Es la atención que resuelve el cuello de botella de la clase 08.
3. Feed-forward.

Al final, una capa lineal convierte cada posición en logits sobre el vocabulario de destino.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# DecoderLayer                             [256, 17, 256]            --
# ├─MultiHeadAttention: 1-1                [256, 17, 256]            --
# │    └─Linear: 2-1                       [256, 17, 256]            65,792
# │    └─Linear: 2-2                       [256, 17, 256]            65,792
# │    └─Linear: 2-3                       [256, 17, 256]            65,792
# │    └─Linear: 2-4                       [256, 17, 256]            65,792
# ├─Dropout: 1-2                           [256, 17, 256]            --
# ├─LayerNorm: 1-3                         [256, 17, 256]            512
# ├─MultiHeadAttention: 1-4                [256, 17, 256]            --
# │    └─Linear: 2-5                       [256, 17, 256]            65,792
# │    └─Linear: 2-6                       [256, 17, 256]            65,792
# │    └─Linear: 2-7                       [256, 17, 256]            65,792
# │    └─Linear: 2-8                       [256, 17, 256]            65,792
# ├─Dropout: 1-5                           [256, 17, 256]            --
# ├─LayerNorm: 1-6                         [256, 17, 256]            512
# ├─FeedForward: 1-7                       [256, 17, 256]            --
# │    └─Sequential: 2-9                   [256, 17, 256]            --
# │    │    └─Linear: 3-1                  [256, 17, 1024]           263,168
# │    │    └─ReLU: 3-2                    [256, 17, 1024]           --
# │    │    └─Linear: 3-3                  [256, 17, 256]            262,400
# ├─Dropout: 1-8                           [256, 17, 256]            --
# ├─LayerNorm: 1-9                         [256, 17, 256]            512
# Total params: 1,053,440


class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, forward_expansion, dropout):
        super(DecoderLayer, self).__init__()
        # TODO: self.self_attention y self.encoder_attention (MultiHeadAttention), self.feed_forward,
        # self.norm1, self.norm2 y self.norm3 (nn.LayerNorm(d_model)) y self.dropout
        pass

    def forward(self, x, enc_out, src_mask, trg_mask):
        # x: [batch_size, trg_len, d_model]
        # enc_out: [batch_size, src_len, d_model]
        # src_mask: [batch_size, 1, 1, src_len]
        # trg_mask: [batch_size, 1, trg_len, trg_len]
        # TODO:
        #   1. self-attention (queries, keys y values = x) con trg_mask; x = norm1(x + dropout(...))
        #   2. atención encoder-decoder: queries = x, keys = values = enc_out, con src_mask;
        #      x = norm2(x + dropout(...))
        #   3. x = norm3(x + dropout(feed_forward(x)))
        pass


rand_enc_out = torch.randn((BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL, D_MODEL))  # salida del encoder al azar
rand_trg_input = torch.randn((BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL, D_MODEL))  # embeddings del target al azar
rand_src_mask = torch.ones((BATCH_SIZE, 1, 1, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)
rand_trg_mask = torch.ones((BATCH_SIZE, 1, MAX_SENTENCE_LENGTH_MODEL, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)

summary(
    DecoderLayer(d_model=D_MODEL, heads=HEADS, forward_expansion=FORWARD_EXPANSION, dropout=DROPOUT),
    input_data=(rand_trg_input, rand_enc_out, rand_src_mask, rand_trg_mask),
)

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# Decoder                                  [256, 17, 17920]          --
# ├─Embedding: 1-1                         [256, 17, 256]            4,587,520
# ├─PositionalEncoding: 1-2                [256, 17, 256]            --
# ├─Dropout: 1-3                           [256, 17, 256]            --
# ├─ModuleList: 1-4                        --                        --
# │    └─DecoderLayer: 2-1                 [256, 17, 256]            1,053,440
# │    └─DecoderLayer: 2-2                 [256, 17, 256]            1,053,440
# │    └─DecoderLayer: 2-3                 [256, 17, 256]            1,053,440
# ├─Linear: 1-5                            [256, 17, 17920]          4,605,440
# Total params: 12,353,280

class Decoder(nn.Module):
    def __init__(self, trg_vocab_size, d_model, num_layers, heads, forward_expansion, dropout, max_length):
        super(Decoder, self).__init__()
        self.d_model = d_model
        # TODO: self.word_embedding (con padding_idx=TRG_VOCAB[PAD_TOKEN]), self.position_embedding,
        # self.dropout, self.layers (nn.ModuleList con num_layers DecoderLayer) y
        # self.fc_out: nn.Linear(d_model, trg_vocab_size)
        pass

    def forward(self, x, enc_out, src_mask, trg_mask):
        # x: [batch_size, trg_len] con índices enteros
        # enc_out: [batch_size, src_len, d_model]
        # TODO:
        #   1. embedding, positional encoding y dropout
        #   2. pasar por cada capa de self.layers
        #   3. devolver self.fc_out(x): logits [batch_size, trg_len, trg_vocab_size]
        pass


rand_enc_out = torch.randn((BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL, D_MODEL))
rand_trg_input = torch.randint(0, TRG_VOCAB_SIZE, (BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL))  # batch de tokens al azar
rand_src_mask = torch.ones((BATCH_SIZE, 1, 1, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)
rand_trg_mask = torch.ones((BATCH_SIZE, 1, MAX_SENTENCE_LENGTH_MODEL, MAX_SENTENCE_LENGTH_MODEL), dtype=torch.bool)

summary(
    Decoder(TRG_VOCAB_SIZE, D_MODEL, NUM_LAYERS, HEADS, FORWARD_EXPANSION, DROPOUT, MAX_SENTENCE_LENGTH_MODEL),
    input_data=(rand_trg_input, rand_enc_out, rand_src_mask, rand_trg_mask),
    depth=2,
)

### Transformer

El modelo completo crea las dos máscaras, pasa el source por el encoder y el target por el decoder. A diferencia de la clase 08, el encoder le entrega al decoder una salida por cada palabra del source, no un único estado.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# Transformer                                   [256, 17, 17920]          --
# ├─Encoder: 1-1                                [256, 17, 256]            --
# │    └─Embedding: 2-1                         [256, 17, 256]            2,961,664
# │    └─PositionalEncoding: 2-2                [256, 17, 256]            --
# │    └─Dropout: 2-3                           [256, 17, 256]            --
# │    └─ModuleList: 2-4                        --                        2,369,280
# ├─Decoder: 1-2                                [256, 17, 17920]          --
# │    └─Embedding: 2-5                         [256, 17, 256]            4,587,520
# │    └─PositionalEncoding: 2-6                [256, 17, 256]            --
# │    └─Dropout: 2-7                           [256, 17, 256]            --
# │    └─ModuleList: 2-8                        --                        3,160,320
# │    └─Linear: 2-9                            [256, 17, 17920]          4,605,440
# Total params: 17,684,224

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        d_model=512,
        num_layers=6,
        forward_expansion=4,
        heads=8,
        dropout=0.1,
        max_length=100,
    ):
        super(Transformer, self).__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        # TODO: self.encoder (Encoder) y self.decoder (Decoder) con los mismos hiperparámetros
        pass

    def forward(self, src, trg):
        # src: [batch_size, src_len]; trg: [batch_size, trg_len]
        # TODO:
        #   1. src_mask = create_src_mask(...) y trg_mask = create_trg_mask(...)
        #   2. pasar src por el encoder -> enc_out: [batch_size, src_len, d_model]
        #   3. devolver la salida del decoder: [batch_size, trg_len, trg_vocab_size]
        pass


summary(
    Transformer(
        SRC_VOCAB_SIZE,
        TRG_VOCAB_SIZE,
        SRC_VOCAB[PAD_TOKEN],
        TRG_VOCAB[PAD_TOKEN],
        d_model=D_MODEL,
        num_layers=NUM_LAYERS,
        forward_expansion=FORWARD_EXPANSION,
        heads=HEADS,
        dropout=DROPOUT,
        max_length=MAX_SENTENCE_LENGTH_MODEL,
    ),
    input_data=(
        torch.randint(0, SRC_VOCAB_SIZE, (BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL)),
        torch.randint(0, TRG_VOCAB_SIZE, (BATCH_SIZE, MAX_SENTENCE_LENGTH_MODEL)),
    ),
    depth=2,
)

## Entrenamiento

> We used the Adam optimizer with $\beta_1 = 0.9$, $\beta_2 = 0.98$ and $\epsilon = 10^{-9}$. We varied the learning rate over the course of training, according to the formula:
>
> $$
> lrate = d_{\text{model}}^{-0.5} \cdot \min(step\_num^{-0.5},\ step\_num \cdot warmup\_steps^{-1.5})
> $$
>
> This corresponds to increasing the learning rate linearly for the first $warmup\_steps$ training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used $warmup\_steps = 4000$.
>
> [...] **Label Smoothing**: During training, we employed label smoothing of value $\epsilon_{ls} = 0.1$. This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.

Al principio del entrenamiento los pesos son aleatorios y los gradientes de la atención son inestables; un learning rate alto en esos primeros pasos puede arruinar el entrenamiento. El *warmup* lo sube de a poco y después lo baja. Lo implementamos con [`LambdaLR`](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LambdaLR.html), que multiplica el learning rate del optimizador por una función del paso; por eso el optimizador se crea con `lr=1.0`. El learning rate se actualiza en cada batch, no en cada época. Usamos `WARMUP_STEPS = 2000` en lugar de 4000 porque entrenamos muchos menos pasos que el paper (unos 9,000 contra 100,000).

In [ ]:
WARMUP_STEPS = 2000
EPOCHS = 12


def lr_schedule(step):
    step = step + 1  # LambdaLR empieza a contar en 0
    return D_MODEL**-0.5 * min(step**-0.5, step * WARMUP_STEPS**-1.5)


steps = range(EPOCHS * len(train_loader))
plt.figure(figsize=(8, 3))
plt.plot(steps, [lr_schedule(s) for s in steps])
plt.xlabel("Paso (batch)")
plt.ylabel("Learning rate")
plt.show()

In [ ]:
torch.manual_seed(SEED)
model = Transformer(
    SRC_VOCAB_SIZE,
    TRG_VOCAB_SIZE,
    SRC_VOCAB[PAD_TOKEN],
    TRG_VOCAB[PAD_TOKEN],
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    forward_expansion=FORWARD_EXPANSION,
    heads=HEADS,
    dropout=DROPOUT,
    max_length=MAX_SENTENCE_LENGTH_MODEL,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=TRG_VOCAB[PAD_TOKEN], label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_schedule)

### Teacher forcing en paralelo

En la clase 08 el decoder recorría el target paso a paso, y en cada paso sorteábamos si recibía el token correcto o su predicción. El Transformer recibe el target completo de una vez: la entrada es `trg[:, :-1]`, el objetivo es `trg[:, 1:]` (el mismo desplazamiento de la clase 08) y la máscara *look-ahead* impide que cada posición vea las siguientes. Así, en una sola pasada, el modelo predice todas las posiciones a la vez, cada una a partir de las palabras correctas anteriores. Es teacher forcing en todos los pasos.

El precio es el *exposure bias* de la clase 08: en inferencia el modelo recibe sus propias predicciones. No usamos *scheduled sampling* porque obligaría a generar paso a paso durante el entrenamiento y se perdería el paralelismo.

La validación también se calcula con teacher forcing, así que las dos pérdidas son comparables entre sí, pero no con la pérdida de validación de la clase 08, que se medía sin teacher forcing. Para comparar los dos modelos usamos BLEU.

In [ ]:
def process_batch(src, trg, model, criterion):
    trg_input = trg[:, :-1]  # entrada del decoder: sin el último token
    trg_target = trg[:, 1:]  # objetivo: sin <SOS>

    # TODO:
    #   1. output = model(src, trg_input) -> [batch_size, trg_len - 1, trg_vocab_size]
    #   2. CrossEntropyLoss espera (N, clases) y (N,): output.reshape(-1, output.size(-1))
    #      y trg_target.reshape(-1)
    #   3. devolver criterion(output, trg_target)
    pass

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, criterion):
    model.train()
    epoch_loss = 0

    for src, trg in dataloader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)

        optimizer.zero_grad()
        batch_loss = process_batch(src, trg, model, criterion)
        batch_loss.backward()
        optimizer.step()
        scheduler.step()  # el learning rate cambia en cada batch

        epoch_loss += batch_loss.item()

    return epoch_loss / len(dataloader)


def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in dataloader:
            src = src.to(DEVICE)
            trg = trg.to(DEVICE)
            epoch_loss += process_batch(src, trg, model, criterion).item()

    return epoch_loss / len(dataloader)

In [ ]:
def train(model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs):
    train_losses, val_losses = [], []
    best_val_loss, best_weights = float("inf"), None

    for epoch in range(n_epochs):
        start = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion)
        val_loss = evaluate(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Guardamos los pesos de la época con menor pérdida de validación
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())

        print(
            f"Época {epoch + 1:02d} | LR: {scheduler.get_last_lr()[0]:.2e} | Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} | {time.time() - start:.0f} s"
        )

    # El modelo queda con los pesos de la mejor época, no de la última
    model.load_state_dict(best_weights)
    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train(
    model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=EPOCHS
)

In [ ]:
plot_training(train_losses, val_losses, mark_min=True)

Esta vez las dos pérdidas se miden con teacher forcing, así que son comparables. En las primeras épocas la pérdida de entrenamiento queda por encima de la de validación por dos motivos: es el promedio de toda la época, que empieza con pesos aleatorios, y en entrenamiento se aplica dropout. Desde la época 5 queda por debajo y la brecha crece despacio (en nuestra ejecución, 2.14 contra 2.35 en la época 12). El learning rate impreso muestra el warmup: sube hasta la época 3 y después baja.

La pérdida de validación todavía baja en la última época, así que con más épocas el modelo probablemente mejore un poco. Ninguna curva se acerca a cero: con label smoothing el mínimo de la pérdida no es cero, y muchas oraciones admiten varias traducciones.

## Generación de traducciones

La generación es autorregresiva y greedy, como en la clase 08: el decoder empieza con `<SOS>` y en cada paso agrega la palabra más probable, hasta generar `<EOS>` o llegar a `max_len` tokens. El encoder se ejecuta una sola vez.

Una diferencia con la clase 08: el decoder no tiene un estado que resuma lo generado, así que en cada paso recibe **todo** el prefijo generado hasta el momento y nos quedamos con la predicción de la última posición. Las implementaciones reales guardan las keys y values de los pasos anteriores (*KV cache*) para no recalcularlos.

In [ ]:
def translate_sentence(model, sentence, max_len):
    model.eval()
    sentence_indexes = encode_sentence(clean_text(sentence), SRC_VOCAB)
    src = torch.tensor(sentence_indexes).unsqueeze(0).to(DEVICE)  # (1, src_len)
    src_mask = create_src_mask(src, SRC_VOCAB[PAD_TOKEN])

    generated_tokens = []
    with torch.no_grad():
        enc_out = model.encoder(src, src_mask)  # el encoder se ejecuta una sola vez
        trg = torch.tensor([[TRG_VOCAB[SOS_TOKEN]]], device=DEVICE)  # (1, 1): <SOS>

        for _ in range(max_len):
            trg_mask = create_trg_mask(trg, TRG_VOCAB[PAD_TOKEN])
            output = model.decoder(trg, enc_out, src_mask, trg_mask)  # (1, largo del prefijo, vocab)
            top1 = output[0, -1].argmax().item()  # greedy, en la última posición

            if top1 == TRG_VOCAB[EOS_TOKEN]:
                break

            generated_tokens.append(top1)
            # autorregresivo: la predicción se agrega al prefijo
            trg = torch.cat([trg, torch.tensor([[top1]], device=DEVICE)], dim=1)

    return decode_sentence(generated_tokens, TRG_VOCAB)

In [ ]:
sentences = [
    "I am hungry",
    "I'm sad",
    "every time I study, I get sleepy",
    "I'm going to the movies",
    "I don't know what to do",
    "I love deep learning",
    "you can go if you want to",
    "where does all this come from?",
    "I can't believe it",
    "Do you want to go to the movies?",
    "let's grab a coffee sometime",
    "she's my best friend",
    "My class is in 30 minutes.",
    "This is a very long sentence, let's see how the model handles it.",
    "I would like to make a reservation for two people at eight o'clock.",
]

for sentence in sentences:
    print(f"Inglés:     {sentence}")
    print(f"Traducción: {translate_sentence(model, sentence, max_len=MAX_SENTENCE_LENGTH_MODEL)}\n")

Comparadas con las de la clase 08, las traducciones de oraciones largas mejoran: en la clase 08 la oración larga se degradaba por completo, y acá conserva la estructura ("esta es una frase muy larga"). Los errores de vocabulario siguen: "deep learning" se traduce palabra por palabra, y el modelo genera `<UNK>`, que aprendió a producir en el entrenamiento para las palabras poco frecuentes. En la última oración el modelo omite "eight" y queda "en punto" sin la hora.

In [ ]:
translate_sentence(model, "Attention is all you need", max_len=MAX_SENTENCE_LENGTH_MODEL)

### ¿A qué atiende el decoder?

En la última llamada al decoder, la atención encoder-decoder de cada capa guardó sus pesos: para cada palabra generada (fila), cuánto peso le dio a cada palabra del source (columna). Graficamos los de la última capa, promediados sobre las cabezas.

In [ ]:
def plot_cross_attention(model, sentence):
    translation = translate_sentence(model, sentence, max_len=MAX_SENTENCE_LENGTH_MODEL)
    # pesos de la última llamada: [1, heads, largo del prefijo, src_len]
    weights = model.decoder.layers[-1].encoder_attention.attention_weights[0].mean(dim=0).cpu()

    src_words = clean_text(sentence).split()
    trg_words = translation.split() + [EOS_TOKEN]  # la fila i predice la palabra i
    trg_words = trg_words[: weights.size(0)]  # si se cortó por max_len, no hay fila de <EOS>

    plt.figure(figsize=(6, 5))
    plt.imshow(weights[: len(trg_words)], cmap="viridis")
    plt.xticks(range(len(src_words)), src_words, rotation=45)
    plt.yticks(range(len(trg_words)), trg_words)
    plt.colorbar()
    plt.title("Atención encoder-decoder (última capa)")
    plt.show()


plot_cross_attention(model, "I would like to make a reservation for two people at eight o'clock.")

Cada palabra generada concentra su atención en la palabra del source que traduce: "reserva" en "reservation", "dos" en "two", "personas" en "people" y "punto" en "clock". Nadie le indicó al modelo qué palabra corresponde a cuál; el alineamiento surge de entrenar para predecir la palabra siguiente. El gráfico también ayuda a entender el error: ninguna fila atiende con claridad a "eight", y la traducción omite la hora. Las otras capas y cabezas pueden atender a otras cosas (ejercicio 4).

## Evaluación en test

Como en la clase 08, traducimos `N_BLEU` oraciones del conjunto de test, una sola vez, con el modelo final, y calculamos BLEU. El paper reporta 27.3 BLEU en inglés-alemán con el modelo base, pero sobre otro dataset y con oraciones más largas: los números no son comparables.

Para comparar con la clase 08 (28.2), calculamos también el BLEU sobre las oraciones de test de hasta 8 palabras en los dos idiomas. Los conjuntos de test no son los mismos, así que la comparación es aproximada.

In [ ]:
N_BLEU = 2000

references, hypotheses, is_short = [], [], []
for src, trg in list(test_pairs)[:N_BLEU]:
    prediction = translate_sentence(model, src, max_len=MAX_SENTENCE_LENGTH_MODEL)
    references.append([trg.split()])  # lista de referencias por oración (acá, una sola)
    hypotheses.append(prediction.split())
    is_short.append(len(src.split()) <= 8 and len(trg.split()) <= 8)

short_refs = [r for r, s in zip(references, is_short) if s]
short_hyps = [h for h, s in zip(hypotheses, is_short) if s]
print(f"BLEU en test ({len(hypotheses)} oraciones): {100 * corpus_bleu(references, hypotheses):.1f}")
print(f"BLEU en test, hasta 8 palabras ({len(short_hyps)} oraciones): {100 * corpus_bleu(short_refs, short_hyps):.1f}\n")

for (src, trg), hyp in list(zip(test_pairs, hypotheses))[:8]:
    print(f"Inglés:     {src}")
    print(f"Referencia: {trg}")
    print(f"Modelo:     {' '.join(hyp)}\n")

En nuestra ejecución el BLEU en test es 45.1, y 47.9 sobre las oraciones de hasta 8 palabras, contra 28.2 del seq2seq con LSTM de la clase 08, con un tiempo de entrenamiento parecido. La comparación es aproximada: este modelo se entrenó con más pares (oraciones de hasta 15 palabras) y los conjuntos de test son distintos. Los ejemplos muestran otra vez los límites de BLEU con una sola referencia: "alquilamos el piso" y "ese coche que está enfrente de tu casa" son traducciones correctas, con palabras de otra variante del español, pero BLEU las penaliza.

## Conclusiones

- El Transformer reemplaza la recurrencia por atención: cada posición combina la información de todas las demás con pesos que dependen de su contenido.
- Como la atención no distingue el orden, el orden se agrega con el positional encoding, que se suma a los embeddings.
- Las máscaras controlan qué puede ver cada posición: la de padding oculta los `<PAD>` y la *look-ahead* oculta las palabras siguientes del target.
- Con el desplazamiento del target y la máscara *look-ahead*, el entrenamiento es teacher forcing sobre todas las posiciones en una sola pasada, sin recorrer la oración paso a paso.
- La generación sigue siendo autorregresiva: un token por vez, con todo el prefijo como entrada.
- La atención encoder-decoder elimina el cuello de botella de la clase 08: el decoder consulta todas las salidas del encoder en cada paso, y la traducción de oraciones largas mejora.

## Ejercicios

1. **Muestreo en vez de greedy**: en lugar de tomar siempre el token más probable, muestrear de la distribución del modelo con [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html). Agregar una *temperatura* (dividir los logits por $T$ antes del softmax) y *top-k sampling* ([`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)). ¿Cómo cambian las traducciones con $T = 0.5$, $1$ y $2$?
2. **Sin warmup**: entrenar con Adam y un learning rate constante (por ejemplo `1e-3`) y comparar las curvas de pérdida con las del schedule del paper.
3. **Pre-LN**: cambiar las subcapas a `x = x + Dropout(Subcapa(LayerNorm(x)))`, la variante que usan la mayoría de los modelos actuales ([Xiong et al., 2020](https://arxiv.org/abs/2002.04745)). ¿Hace falta el warmup?
4. **Atención por cabeza**: graficar los pesos de la atención encoder-decoder de cada cabeza por separado. ¿Las cabezas atienden a cosas distintas?
5. **Compartir pesos**: el paper comparte la matriz del embedding del decoder con la capa lineal de salida (*weight tying*). Implementarlo y comparar la cantidad de parámetros y el BLEU.

#### Lectura adicional

- Vaswani et al. (2017), [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762): el paper que replicamos.
- Bahdanau et al. (2015), [*Neural Machine Translation by Jointly Learning to Align and Translate*](https://arxiv.org/abs/1409.0473): la atención sobre un seq2seq con RNN, antes del Transformer.
- [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) (Harvard NLP): el paper completo, línea por línea, con código en PyTorch.
- Devlin et al. (2019), [*BERT*](https://arxiv.org/abs/1810.04805) y Radford et al. (2018), [*GPT*](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf): modelos que usan solo el encoder o solo el decoder del Transformer.